# 🎯 Notebook 04 — Feature Selection

**Objective**: Reduce dimensionality and select the most predictive features for the fraud model.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

FEATURE_DIR = os.path.join('..', 'data', 'features')
txn = pd.read_parquet(os.path.join(FEATURE_DIR, 'transactions_features.parquet'))

print(f"Loaded dataset with {txn.shape[1]} features.")

Loaded dataset with 318 features.


In [2]:
# We will drop IDs and target for feature selection
drop_cols = ['transaction_id', 'customer_cif_id', 'customer_account_number', 'device_id_fingerprint', 'wallet_account_id', 'is_aml', 'aml_typology', 'fraud_intensity_score', 'fis_band']
features = [c for c in txn.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(txn[c])]

X = txn[features].fillna(0)
y = txn['is_aml']

print(f"Selecting from {len(features)} numerical features...")
# Using a fast method: correlation with target or Random Forest Feature Importance
# Here we will use a small sample to compute RF feature importance for speed
X_sample = X.sample(n=min(50000, len(X)), random_state=42)
y_sample = y.loc[X_sample.index]

rf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_sample, y_sample)

importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
top_50_features = importances.head(50).index.tolist()
print("Top 10 features:")
print(importances.head(10))

# Save selected features dataset
selected_cols = drop_cols + top_50_features
txn[selected_cols].to_parquet(os.path.join(FEATURE_DIR, 'transactions_selected.parquet'), index=False)
print("Saved feature-selected dataset.")

Selecting from 225 numerical features...


Top 10 features:
transaction_amount                        0.218290
sender_running_balance_txn_amount         0.193071
sender_cumulative_daily_balance_change    0.081894
typology_signal                           0.067309
ip_flag_country_high_risk                 0.020230
receiver_current_balance                  0.017962
sender_current_balance                    0.016011
debit_summation_period                    0.014831
credit_summation_period                   0.014345
ip_flag_cross_border                      0.013388
dtype: float64


Saved feature-selected dataset.
